In [17]:
import sys
print(sys.executable)

/Users/samaalshehri/Developer/.venv/bin/python


In [18]:
import duckdb, os
print(os.getcwd())  
con = duckdb.connect("rfm.duckdb")
con.sql("""
    CREATE OR REPLACE TABLE sales AS
    SELECT * FROM read_csv_auto('_data/online_retail.csv')
""")
con.sql("DESCRIBE sales").df()

/Users/samaalshehri/Developer/projects/online_retail


,column_name,column_type,null,key,default,extra
0,Invoice,VARCHAR,YES,None,None,None
1,StockCode,VARCHAR,YES,None,None,None
2,Description,VARCHAR,YES,None,None,None
3,Quantity,BIGINT,YES,None,None,None
4,InvoiceDate,TIMESTAMP,YES,None,None,None
5,Price,DOUBLE,YES,None,None,None
6,Customer ID,DOUBLE,YES,None,None,None
7,Country,VARCHAR,YES,None,None,None


In [19]:
con.sql("SELECT COUNT(*) AS total_rows FROM sales").df()

,total_rows
0,1067371


In [20]:
con.sql("SELECT MIN(InvoiceDate) AS first_date, MAX(InvoiceDate) AS last_date FROM sales").df()

,first_date,last_date
0,2009-12-01 07:45:00,2011-12-09 12:50:00


In [21]:
con.sql('SELECT COUNT(*) AS no_customer FROM sales WHERE "Customer ID" IS NULL').df()

,no_customer
0,243007


In [22]:
con.sql("SELECT COUNT(*) AS cancellations FROM sales WHERE Invoice LIKE 'C%'").df()

,cancellations
0,19494


In [23]:
con.sql("SELECT COUNT(*) AS bad_rows FROM sales WHERE Quantity <= 0 OR Price <= 0").df()

,bad_rows
0,25700


22.8% of rows have no customer ID, removed because RFM needs a customer

In [24]:
con.sql(""" 
create or replace table clean_sales AS 
select 
 invoice, 
 invoicedate, 
 CAST("Customer ID" AS INTEGER) AS customer_id,
 Quantity,
 Price, 
 Quantity * Price AS revenue

from sales 
 where
   "Customer ID" IS NOT NULL
     and Quantity > 0 
     and Price > 0 
     and Invoice not like 'C%'
 """)


CountBIGINT
805549


In [ ]:
con.sql(""" 
select COUNT(*) AS rows_left, 
    COUNT(DISTINCT customer_id) AS customers,
    min(revenue) AS min_revenue
from clean_sales
  """).df()

,rows_left,customers,min_revenue
0,805549,5878,0.001
